In [ ]:
!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
import os, re, zipfile, gc
import pandas as pd
import numpy as np
import torch
from torch import nn
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

# ── Reproducibility ──────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── CUDA memory fragmentation fix ────────────────────────────────
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ── Hyper-parameters (edit here) ─────────────────────────────────
MODEL_NAME   = "xlm-roberta-base"
MAX_LEN      = 512
NUM_EPOCHS   = 15
BATCH_SIZE   = 4
EVAL_BATCH   = 8
GRAD_ACCUM   = 2        # effective batch = BATCH_SIZE × GRAD_ACCUM
LR           = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
FP16         = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ── Data paths (change to match your environment) ────────────────
TRAIN_PATH = "/content/data/english/train_data.csv"
TEST_PATH  = "/content/data/english/test_data.csv"

In [ ]:
def light_clean(text: str) -> str:
    """Strip HTML tags, URLs, and excess whitespace."""
    text = re.sub(r"<[^>]+>", "", str(text))
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# ── Label maps ───────────────────────────────────────────────────
regret_map = {"No Regret": 0, "Action": 1, "Inaction": 2}
regret_rev = {v: k for k, v in regret_map.items()}

domain_map = {
    "Romance friends and parents": 0,
    "Other domains": 1,
    "Education": 2,
    "Career and Finance": 3,
    "Health": 4,
}
domain_rev = {v: k for k, v in domain_map.items()}

# ── Read CSVs ────────────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

# ── Combine title + text ─────────────────────────────────────────
train_df["full_text"] = (
    train_df["title"].fillna("") + " - " + train_df["text"].fillna("")
).apply(light_clean)
test_df["full_text"] = (
    test_df["title"].fillna("") + " - " + test_df["text"].fillna("")
).apply(light_clean)

# ── Encode labels ────────────────────────────────────────────────
train_df["regret_label"] = train_df["Regret"].map(regret_map)
train_df["domain_label"] = train_df["Domain"].map(domain_map)

print(f"Train samples: {len(train_df)}  |  Test samples: {len(test_df)}")
print(train_df[["Regret", "regret_label"]].value_counts())
print()
print(train_df[["Domain", "domain_label"]].value_counts())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class ReDDITDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])


def compute_metrics(pred):
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds),
    }


def make_weighted_trainer(model, train_ds, val_ds, weights_tensor, out_dir):
    """Trainer that applies per-class weights in CrossEntropyLoss."""

    class WeightedTrainer(Trainer):
        def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
            labels  = inputs.pop("labels")
            outputs = model(**inputs)
            loss_fct = nn.CrossEntropyLoss(weight=weights_tensor.to(outputs.logits.device))
            loss = loss_fct(
                outputs.logits.view(-1, model.config.num_labels),
                labels.view(-1),
            )
            return (loss, outputs) if return_outputs else loss

    args = TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=3,
        fp16=FP16,
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        report_to="none",
    )

    return WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
    )

In [ ]:
# ── Split ────────────────────────────────────────────────────────
tr_txt, val_txt, tr_lab, val_lab = train_test_split(
    train_df["full_text"].tolist(),
    train_df["regret_label"].tolist(),
    test_size=0.1,
    stratify=train_df["regret_label"].tolist(),
    random_state=SEED,
)

train_ds_r = ReDDITDataset(tr_txt, tr_lab)
val_ds_r   = ReDDITDataset(val_txt, val_lab)

# ── Class weights ────────────────────────────────────────────────
w_regret = torch.tensor(
    compute_class_weight(
        "balanced",
        classes=np.unique(train_df["regret_label"]),
        y=train_df["regret_label"],
    ),
    dtype=torch.float32,
).to(DEVICE)

# ── Model & Trainer ──────────────────────────────────────────────
model_regret = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3
).to(DEVICE)

trainer_regret = make_weighted_trainer(
    model_regret, train_ds_r, val_ds_r, w_regret, "./results_regret"
)
trainer_regret.train()

In [ ]:
# ── Free GPU memory first ────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()

# ── Drop rows without domain label ───────────────────────────────
dom_df = train_df.dropna(subset=["domain_label"])

tr_txt_d, val_txt_d, tr_lab_d, val_lab_d = train_test_split(
    dom_df["full_text"].tolist(),
    dom_df["domain_label"].astype(int).tolist(),
    test_size=0.1,
    stratify=dom_df["domain_label"].astype(int).tolist(),
    random_state=SEED,
)

train_ds_d = ReDDITDataset(tr_txt_d, tr_lab_d)
val_ds_d   = ReDDITDataset(val_txt_d, val_lab_d)

w_domain = torch.tensor(
    compute_class_weight(
        "balanced",
        classes=np.unique(dom_df["domain_label"]),
        y=dom_df["domain_label"],
    ),
    dtype=torch.float32,
).to(DEVICE)

model_domain = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=5
).to(DEVICE)

trainer_domain = make_weighted_trainer(
    model_domain, train_ds_d, val_ds_d, w_domain, "./results_domain"
)
trainer_domain.train()

In [ ]:
test_ds = ReDDITDataset(test_df["full_text"].tolist())

regret_preds = trainer_regret.predict(test_ds).predictions.argmax(-1)
domain_preds = trainer_domain.predict(test_ds).predictions.argmax(-1)

sub_df = pd.DataFrame({
    "id":     test_df["id"],
    "Regret": [regret_rev[p] for p in regret_preds],
    "Domain": [domain_rev[p] for p in domain_preds],
})

sub_df.to_csv("predictions.csv", index=False)

with zipfile.ZipFile("submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write("predictions.csv", arcname="predictions.csv")

print("✅  submission.zip ready")
print(sub_df.head())